In [17]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
from tqdm import tqdm

In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
])
val_dataset = Imagenette(root = './data', split = 'val', download = True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False, num_workers = 4)

In [23]:
model = model.to(DEVICE)
model.eval()

if torch.cuda.is_available():
  model = model.half()


all_classes = [label[0] for label in val_dataset.classes]
correct = 0
total = 0
with torch.no_grad():
  for images, labels in tqdm(val_dataset):
    images = images.to(DEVICE)
    pil_images = [torchvision.transforms.ToPILImage()(image) for image in images]
    inputs = processor(text = all_classes, images = pil_images, return_tensors = "pt", padding = True).to(DEVICE)
    
    outputs = model(**inputs)
    predicted = outputs.logits_per_image.argmax(dim=-1)
    total += len(images)
    correct += (predicted == labels).sum().item()
accuracy = (correct / total)
print(f"Accuracy Score: ", accuracy)
print(f"Correct / Total: {correct} / {total}")

  0%|          | 0/3925 [00:00<?, ?it/s]

100%|██████████| 3925/3925 [09:31<00:00,  6.87it/s]

Accuracy Score:  0.9677282377919321
Correct / Total: 11395 / 11775
